In [ ]:
from datetime import datetime
semana=datetime.today().strftime('%U')
semana_atual= int(semana) + 1
print(semana_atual)

# Coletando informações lógicas

In [ ]:
import pandas as pd

arquivo_GNBDUFunction= 'C:/Camilo/' + '5G_GNBDUFunction'+ '_'+str(semana_atual)+'.txt'
arquivo_NRCellDU= 'C:/Camilo/' + '5G_NRCellDU'+ '_'+str(semana_atual)+'.txt'


GNBDUFunction= pd.read_csv(arquivo_GNBDUFunction, delimiter='\t', engine='python')
NRCellDU= pd.read_csv(arquivo_NRCellDU, delimiter='\t', engine='python')

GNBDUFunction1= GNBDUFunction.loc[:, ['eNB', 'gNBId']]
NRCellDU1= NRCellDU.loc[:, ['UF', 'eNB', 'nRCellDUId', 'cellLocalId', 'administrativeState', 'cellState', 
                            'nCI', 'nRPCI', 'nRTAC'
, 'pLMNIdList_mcc', 'pLMNIdList_mnc', 'pointAArfcnTdd', 'ssbFrequency']] 

junta= pd.merge(NRCellDU1,GNBDUFunction1, left_on='eNB', right_on= 'eNB')
junta.dropna(subset=['gNBId'],inplace=True)

junta['CGI'] = (junta['pLMNIdList_mcc'].astype(str).str[0:3].astype('int').astype(str) + 
                junta['pLMNIdList_mnc'].astype(str).str[0:2].astype('int').astype(str) + 
                '-' + junta['gNBId'].astype('int').astype(str) + 
                '-' + junta['cellLocalId'].astype('int').astype(str))
#junta['Band']= '700'
#junta['Band']= junta['DL_NR_ARFCN'].replace(['156664', '432044', '470184', '626724', '628062', '633390'],
#['700', '2100','2300', '3400', '3400', '3500'])
# v3: a chave do site sai do NOME DA CELULA, nao do no. Em Hotel de BTS o no
# hospeda sites remotos, e a chave pelo no fazia o hospede herdar a coordenada,
# o endereco e o Site_Name do hospedeiro (medido: 102 de 102 hospedes errados).
# Em site normal nRCellDUId[2:5] == eNB[1:4], entao a chave e IDENTICA a da v2.
junta['Aux_NR']= junta['nRCellDUId'].str[2:5]+junta['eNB'].str[-3:]+'_'+junta['nRCellDUId'].str[-1:]
# junta['DL_NR_ARFCN']= junta['pointAArfcnDlFdd'].fillna('').astype(str)+junta['pointAArfcnTdd'].fillna('').astype(str)

junta['DL_NR_ARFCN']= junta['ssbFrequency'].astype(int)
 


junta.sample(5)

#junta.to_csv('C:/Camilo/junta.csv')

In [ ]:
junta.rename(columns={'gNBId': 'GNodeBID', 'nRCellDUId': 'SectorID',
                        'nRPCI': 'NR_PCI', 'pLMNIdList_mcc': 'MCC', 'pLMNIdList_mnc': 'MNC',
                       'nRTAC': 'NR_TrackingAreaCode', 'nCI':'NCI'},inplace= True)

junta.head()

# Criando dados Science

In [ ]:
import pandas as pd
import glob
import os

# Caminho da pasta
pasta = 'C:/cellref/NR_dump'

# Lista todos os arquivos .xlsx na pasta
arquivos_xlsx = glob.glob(os.path.join(pasta, '*.xlsx'))

# Lista para armazenar os DataFrames
dfs = []

# Lê cada arquivo Excel com cabecalho na linha 3
for arquivo in arquivos_xlsx:
    print(f"Lendo arquivo: {arquivo}")
    df = pd.read_excel(arquivo, header=2)
    dfs.append(df)

# Concatena todos os DataFrames
if dfs:
    science = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal de arquivos concatenados: {len(dfs)}")
    print(f"Total de linhas no DataFrame final: {len(science)}")
else:
    print("Nenhum arquivo .xlsx encontrado na pasta!")
    science = pd.DataFrame()

# science= pd.read_excel('C:/cellref/NR/NR_SP_2.xlsx')

science['COUNTRY']= 'BRAZIL'
science['MARKET']= 'VIVO'
science['SiteID']= science['SIG_ERB']+'_'+science['SIG_ESTADO']
science['Cell_Name']= science['SIG_LOGICA']

science['REGION'] = science['SIG_ESTADO'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'],
                                                 ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

science['COBERTURA_SETOR'] = science['COBERTURA_SETOR'].replace(['AMBOS', 'OUTDOOR', 'INDOOR'],['Outdoor', 'Outdoor',
                                                                                                          'Indoor'])

science.rename(columns={'NOME_MUNICIPIO': 'CLUSTER', 'NOM_ERB': 'Site_Name', 'NUM_ENB_ID': 'eNodeB_ID',
                        'SIG_LOGICA': 'Sector', 'NUM_AZIMUTE_ANTENA': 'Azimuth', 'NUM_TILT_MECAN_ANTENA': 'MechanicalTilt',
                       'NUM_TILT_ELETRICO': 'ElectricalTilt', 'DES_MODELO_ANTENA': 'AntennaType', 'FORNECEDOR': 'Vendor',
                       'COBERTURA_SETOR': 'SiteType', 'NUM_ALT_MAX_SUP_ANTENA': 'Height'
                       , 'STR_LATITUDE_SITE': 'Latitude', 'STR_LONGITUDE_SITE': 'Longitude','DES_ENDERECO_SITE': 'ENDERECO','NOM_BAIRRO': 'BAIRRO', 'AREA_REGISTRO': 'CN'},inplace= True)


science1= science.loc[:,['COUNTRY', 'REGION','MARKET','CN','CLUSTER','BAIRRO','ENDERECO','SiteID','Site_Name','eNodeB_ID','Longitude',
                   'Latitude','Sector','Cell_Name', 'Azimuth', 'MechanicalTilt', 'ElectricalTilt', 
                   'AntennaType', 'Vendor', 'SiteType', 'Height']]


# Funcao para converter Latitude/Longitude

def convert_coordinates(value):
    try:
        if value is None:
            return None
        s = str(value).strip()
        if s == '' or s.lower() == 'nan':
            return None

        # Formato com separador: -DD-MM-SS ou DD-MM-SS
        if s.count('-') >= 2:
            sign = -1 if s.startswith('-') else 1
            s = s.lstrip('-')
            parts = s.split('-')
            degrees = int(parts[0])
            minutes = int(float(parts[1]))
            seconds = float(parts[2].replace(',', '.'))
        else:
            # Formato compacto com hemisferio, ex: 06S431820 ou 038W382480
            import re

            m = re.match(r'^(\d{2,3})([NSEW])(\d{2})(\d{2})(\d{2})$', s.upper())
            if not m:
                return None
            degrees = int(m.group(1))
            hemisphere = m.group(2)
            minutes = int(m.group(3))
            seconds = int(m.group(4)) + (int(m.group(5)) / 100.0)
            sign = -1 if hemisphere in ('S', 'W') else 1

        # Normaliza segundos/minutos caso exceda
        if seconds >= 60:
            minutes += int(seconds // 60)
            seconds = seconds % 60
        if minutes >= 60:
            degrees += int(minutes // 60)
            minutes = minutes % 60

        decimal = sign * (degrees + minutes / 60 + seconds / 3600)
        return decimal
    except Exception as e:
        print(f"Erro ao processar valor: {value}. Detalhes: {e}")
        return None


# Aplicando a funcao para converter Latitude e Longitude

science1['Latitude'] = science1['Latitude'].apply(convert_coordinates)
science1['Longitude'] = science1['Longitude'].apply(convert_coordinates)


# Arredonda e converte para Int64 (aceita NaN)
science1['MechanicalTilt'] = pd.to_numeric(
    science1['MechanicalTilt'].astype(str).str.replace(',', '.'), 
    errors='coerce'
).round().astype('Int64')

science1['ElectricalTilt'] = pd.to_numeric(
    science1['ElectricalTilt'].astype(str).str.replace(',', '.'), 
    errors='coerce'
).round().astype('Int64')  # o 'I' maiúsculo trata N/A como N/A evitando erros

science1['Height'] = pd.to_numeric(
    science1['Height'].astype(str).str.replace(',', '.'), 
    errors='coerce'
).round().astype('Int64')

science1.sample(2)

In [ ]:
#science1['Height']= science1['Height'].replace([0,1,2,3,4,5], [6,6,6,6,6,6])
"""
science1[science1['Height']<6]=0
science1[science1['Azimuth']==360]=0
science1[science1['MechanicalTilt']==360]=0
"""
science1['Height'] = pd.to_numeric(science1['Height'], errors='coerce')
science1['Height'] = science1['Height'].apply(lambda x: 6 if pd.isna(x) else (x if x > 6 else 6))
science1['Azimuth'] = science1['Azimuth'].apply(lambda x: 0 if pd.isna(x) else (x if x != 360 else 0))
science1['MechanicalTilt']=science1['MechanicalTilt'].apply(lambda x: x if x != 360 else 0)
science1['Beamwidth']= 65
science1['CableLossDL']= 0

#science1['MechanicalTilt'].value_counts()
df1= science1.loc[:, ['SiteID','Cell_Name','COUNTRY','REGION', 'MARKET', 'CN','CLUSTER','BAIRRO','ENDERECO','Site_Name', 'Longitude', 'Latitude', 'Azimuth', 'Beamwidth',
               'MechanicalTilt', 'ElectricalTilt', 'AntennaType', 'Vendor', 'SiteType', 'CableLossDL', 'Height']]
#df.loc[df['earfcndl']=='9510',:]

df1.head(2)

# Juntando CellRef e EPRE

In [ ]:
import numpy as np

arquivo_NRSectorCarrier= 'C:/Camilo/' + '5G_NRSectorCarrier'+ '_'+str(semana_atual)+'.txt'

power= pd.read_csv(arquivo_NRSectorCarrier, sep='\t',usecols=['eNB', 'NRSectorCarrier', 'maxTransmissionPower'],low_memory=False)

# NRSectorCarrier.maxTransmissionPower vem em mW (escala linear). No LTE o atributo
# equivalente, SectorCarrier.maximumTransmissionPower, vem em 0,1 dBm (escala logaritmica)
# e por isso la a conversao e /10. Aqui a conversao correta e mW -> dBm, para a coluna EPRE
# ficar na mesma unidade do CellRef LTE. Ex.: 200000 mW = 200 W = 53,01 dBm.
power['maxTransmissionPower']= pd.to_numeric(power['maxTransmissionPower'], errors='coerce')
power['EPRE']= (10 * np.log10(power['maxTransmissionPower'].where(power['maxTransmissionPower'] > 0))).round(2)

power.rename(columns={'eNB': 'eNB_Pwr', 'NRSectorCarrier': 'CELL_Pwr'}, inplace=True)

df3= pd.merge(df1,power, left_on=['SiteID', 'Cell_Name'], right_on=['eNB_Pwr', 'CELL_Pwr'],how='left')

df4= df3.loc[:, ['SiteID','Cell_Name','COUNTRY','REGION', 'MARKET', 'CN', 'CLUSTER','BAIRRO','ENDERECO','Site_Name', 'Longitude', 'Latitude', 'Azimuth', 'Beamwidth',
               'MechanicalTilt', 'ElectricalTilt', 'AntennaType', 'Vendor', 'SiteType', 'EPRE','CableLossDL', 'Height']]
df4.sample(2)

In [ ]:
# v3: NAO alterado. df4 vem da science NR, que ja registra o hospede sob a ERB
# PROPRIA dele (ex.: MUM34_AM para o site UM3), entao SiteID[1:4] ja devolve o
# site fisico correto. Medido: SiteID[1:4] != Cell_Name[2:5] em 12 de 16.639
# linhas (0,07%), e as 12 sao erro de cadastro (linhas deslocadas), que nenhuma
# das duas chaves conserta. O defeito do hotel esta so no lado do DUMP (celula 2).
df4['Aux']= df4['SiteID'].str[1:4]+df4['SiteID'].str[-3:]+'_'+df4['Cell_Name'].str[-1:]

df4.head(2)

# Juntando CellRef com Dados Lógicos

In [ ]:
# Alterei para 'left' para visualizar sites que nao estao cadastrados no science
cellref= pd.merge(junta, df4, left_on= 'Aux_NR', right_on='Aux', how='left')
cellref['Technology_Name']= '5G NR'
cellref['Category_Name']= 'NR_B700'
cellref['Cellname']= cellref['SectorID']

#cellref.drop(['UF','SiteID','cellLocalId','administrativeState','cellState','pointAArfcnUlFdd','CGI','Aux_NR','Cell_Name','Aux'], axis='columns', inplace=True)

# Vou deixar o administrativeState e o cellState

cellref.drop(['UF','SiteID','cellLocalId','CGI','Aux_NR','Cell_Name','Aux'], axis='columns', inplace=True)

cellref.drop_duplicates(subset=['eNB',"SectorID"],inplace=True)
# NAO recuperar o NCI do NRCellCU: nCI vazio no DU == cellState IDLE (correlacao
# perfeita, medida na semana 36 — 80 de 80). Sao sites LOCKED ou recem-criados,
# ainda nao ativados. Trazer para o cadastro faz o Neighbor Viewer mostra-los
# como se ja desse para trabalhar com eles. Decisao do usuario em 02/09/2026:
# celula nao ativada fica FORA. Nao reintroduzir a reserva pelo CU.
cellref.dropna(subset=['NCI'], inplace=True)

cellref.rename(columns={'eNB': 'SiteID'},inplace= True)

cellref['Band']= cellref['Cellname'].str[0:2].replace({
                '5S':3400, '5O':2300, '5U':2100, '5Z':700, '5R':2300
            }).fillna('')


cellref.sample(5)

In [ ]:
cellref['UF']= cellref['SiteID'].str[-2:]

cellref1= cellref.loc[:,['COUNTRY', 'REGION','UF','MARKET','CN','CLUSTER','BAIRRO','ENDERECO','SiteID','Site_Name', 'Cellname','GNodeBID', 'Longitude',
                        'Latitude','SectorID','Cellname','administrativeState','cellState','NCI','DL_NR_ARFCN','NR_PCI','MCC','MNC',
                        'NR_TrackingAreaCode', 'Azimuth', 'Beamwidth', 'MechanicalTilt','ElectricalTilt',
                        'AntennaType','Vendor','Band','SiteType','EPRE','CableLossDL', 'Height','Technology_Name',
                        'Category_Name']]
cellref1.set_index('COUNTRY',inplace=True)
cellref1.rename(columns={'CLUSTER':'MUNICIPIO'},inplace=True)

from datetime import datetime
from pathlib import Path
import numpy as np


def circular_azimuth_mode(series):
    """Moda do azimute; em empate escolhe o candidato de menor dispersao circular."""
    valid = pd.to_numeric(series, errors='coerce').dropna() % 360
    if valid.empty:
        return np.nan
    counts = valid.value_counts()
    candidates = counts[counts == counts.max()].index.to_numpy(dtype=float)
    if len(candidates) == 1:
        return float(candidates[0])
    values = valid.to_numpy(dtype=float)

    def dispersion(candidate):
        return float(np.abs((values - candidate + 180) % 360 - 180).sum())

    return float(min(candidates, key=lambda value: (dispersion(value), value)))


def fill_from_reference(frame, reference, keys, columns, suffix):
    """Preenche apenas vazios, preservando todos os valores existentes."""
    renamed = reference[keys + columns].rename(
        columns={column: f'{column}{suffix}' for column in columns}
    )
    frame = frame.merge(renamed, how='left', on=keys, sort=False)
    for column in columns:
        frame[column] = frame[column].fillna(frame[f'{column}{suffix}'])
    return frame.drop(columns=[f'{column}{suffix}' for column in columns])


output_date = datetime.now().strftime('%m%d%Y')

# 1) Fotografia original do cadastro, mantida para auditoria/notificacao.
original_df = cellref1.copy()
original_path = f'C:/cellref/NR/Cellref_5G_v3_{output_date}.xlsx'
original_df.to_excel(original_path, index=None)

# 2) Referencia operacional do Tool Box.
custom_df = original_df.copy()
# O notebook legado seleciona Cellname duas vezes. O Excel materializa a segunda como
# Cellname.1; explicita-se aqui o mesmo esquema para evitar selecao ambigua no pandas.
seen_columns = {}
unique_columns = []
for column in custom_df.columns:
    occurrence = seen_columns.get(column, 0)
    unique_columns.append(column if occurrence == 0 else f'{column}.{occurrence}')
    seen_columns[column] = occurrence + 1
custom_df.columns = unique_columns
original_columns = custom_df.columns.tolist()
custom_df['_PhysicalSite'] = custom_df['UF'].astype('string') + '|' + custom_df['Cellname'].astype('string').str[2:5]
custom_df['_HostSite'] = custom_df['UF'].astype('string') + '|' + custom_df['SiteID'].astype('string').str[1:4]
# Setor NR normalizado para 1/2/3 (4->1, 5->2, 6->3, ...), igual a referencia LTE,
# para que o co-setor 4G/5G case corretamente antes da moda.
_nr_ultimo_digito = pd.to_numeric(custom_df['Cellname'].astype('string').str[-1], errors='coerce')
custom_df['_SectorKey'] = (((_nr_ultimo_digito - 1) % 3) + 1).astype('Int64').astype('string')
custom_df['_IsHotel'] = custom_df['_PhysicalSite'].ne(custom_df['_HostSite'])

# Primeiro completa pelo proprio cadastro NR: dados geograficos sao comuns ao site.
nr_site_columns = ['REGION', 'MARKET', 'CN', 'MUNICIPIO', 'BAIRRO', 'ENDERECO',
                   'Site_Name', 'Longitude', 'Latitude', 'Vendor', 'SiteType']
nr_site_reference = (
    custom_df.groupby('_PhysicalSite', dropna=False)[nr_site_columns]
    .first().reset_index()
)
custom_df = fill_from_reference(
    custom_df, nr_site_reference, ['_PhysicalSite'], nr_site_columns, '_NR'
)

# O LTE tratado e a referencia preferencial. Se ainda nao foi gerado, usa o CellRef
# LTE datado mais recente; a moda LTE abaixo continua protegendo o fallback de azimute.
lte_custom_path = Path('C:/cellref/LTE/cellref.xlsx')
if lte_custom_path.exists():
    lte_path = lte_custom_path
else:
    lte_candidates = list(Path('C:/cellref/LTE').glob('CellRef_LTE_*.xlsx'))
    if not lte_candidates:
        raise FileNotFoundError('Nenhum CellRef LTE encontrado em C:/cellref/LTE')
    lte_path = max(lte_candidates, key=lambda path: path.stat().st_mtime)
    print(f'AVISO: cellref.xlsx nao encontrado; usando {lte_path.name}')

lte_reference = pd.read_excel(lte_path)
lte_reference['_PhysicalSite'] = (
    lte_reference['UF'].astype('string') + '|' + lte_reference['CELL'].astype('string').str[1:4]
)
lte_raw_sector = pd.to_numeric(
    lte_reference['CELL'].astype('string').str[-1], errors='coerce'
)
lte_reference['_SectorKey'] = (((lte_raw_sector - 1) % 3) + 1).astype('Int64').astype('string')

# Coordenadas/endereco sao comuns ao site fisico. Nunca usa o eNB/gNB hospedeiro:
# em Hotel de BTS a chave fisica vem do nome da celula remota.
lte_site_source = lte_reference.rename(columns={'CLUSTER': 'MUNICIPIO'})
lte_site_columns = ['CN', 'MUNICIPIO', 'BAIRRO', 'ENDERECO', 'Longitude', 'Latitude']
lte_site_reference = (
    lte_site_source.groupby('_PhysicalSite', dropna=False)[lte_site_columns]
    .first().reset_index()
)
custom_df = fill_from_reference(
    custom_df, lte_site_reference, ['_PhysicalSite'], lte_site_columns, '_LTE'
)

# Azimute LTE por co-setor: inclui todas as portadoras 4G e normaliza setores 4/5/6
# para 1/2/3 antes de calcular a moda circular.
lte_sector_reference = (
    lte_reference.groupby(['_PhysicalSite', '_SectorKey'], dropna=False)['Azimuth']
    .agg(circular_azimuth_mode)
    .rename('_LTEAzimuth')
    .reset_index()
)
custom_df = custom_df.merge(
    lte_sector_reference,
    how='left',
    on=['_PhysicalSite', '_SectorKey'],
    sort=False,
)
# Co-setor 4G/5G tem sempre o mesmo azimute, INCLUSIVE em Hotel de BTS: a chave
# _PhysicalSite vem do nome da celula remota, entao o co-setor 4G do site remoto e
# recuperado corretamente mesmo quando o gNB hospedeiro e outro. Onde existe co-setor
# 4G, o 5G recebe o azimute do 4G (autoritativo: 4G tem mais portadoras e moda robusta).
_tem_co_setor_4g = custom_df['_LTEAzimuth'].notna()
custom_df.loc[_tem_co_setor_4g, 'Azimuth'] = custom_df.loc[
    _tem_co_setor_4g, '_LTEAzimuth'
]

# Setores 5G sem co-setor 4G (5G-only e nao-hotel): harmoniza pela moda entre as
# proprias portadoras 5G. Quem ja recebeu o azimute do 4G (inclusive hoteis) fica intacto.
nr_modal_reference = (
    custom_df.loc[~custom_df['_IsHotel']]
    .groupby(['_PhysicalSite', '_SectorKey'], dropna=False)['Azimuth']
    .agg(circular_azimuth_mode)
    .rename('_NRAzimuthModal')
    .reset_index()
)
custom_df = custom_df.merge(
    nr_modal_reference,
    how='left',
    on=['_PhysicalSite', '_SectorKey'],
    sort=False,
)
_sem_co_setor_4g_normal = custom_df['_LTEAzimuth'].isna() & ~custom_df['_IsHotel']
custom_df.loc[_sem_co_setor_4g_normal, 'Azimuth'] = custom_df.loc[
    _sem_co_setor_4g_normal, '_NRAzimuthModal'
].fillna(custom_df.loc[_sem_co_setor_4g_normal, 'Azimuth'])

custom_df.drop(
    columns=['_PhysicalSite', '_HostSite', '_SectorKey', '_IsHotel',
             '_LTEAzimuth', '_NRAzimuthModal'],
    inplace=True,
)
custom_df = custom_df[original_columns]

# v3 PROMOVIDO em 02/09/2026: grava no nome oficial. A comparacao contra a v2 foi
# feita antes de promover — site normal 0 mudancas em 14.074 celulas, 178 hospedes
# de hotel corrigidos. Copia do arquivo pre-correcao em cellref_5g_backup_20260902_pre_hotel.xlsx
custom_path = 'C:/cellref/NR/cellref_5g.xlsx'
custom_df.to_excel(custom_path, index=None)

print(f'Original: {original_path} ({len(original_df):,} linhas)')
print(f'Tool Box: {custom_path} ({len(custom_df):,} linhas)')
